#### networking (test)

In [ ]:
///- --test static --symbol

:< testing

In [1]:
:< rust_operators

:! reqwest_response =
    `(
        rust::rust_type
            `($'reqwest_Response')
            `($'reqwest_wasm::Response')
            "type reqwest_Response = class end"
    )

:! reqwest_error =
    `(
        rust::rust_type
            `($'reqwest_Error')
            `($'reqwest_wasm::Error')
            "type reqwest_Error = class end"
    )

:== request_builder_switch =
    {
        Gleam : $'reqwest_RequestBuilder'
        Lua : $'reqwest_RequestBuilder'
        Fsharp : $'reqwest_RequestBuilder'
        Python : $'reqwest_RequestBuilder'
        TypeScript : $'reqwest_RequestBuilder'
        Cpp : $'reqwest_RequestBuilder'
        CppHost : $'reqwest_RequestBuilder'
        Pascal : $'reqwest_RequestBuilder'
        Cuda : $'reqwest_RequestBuilder'
        C : $'reqwest_RequestBuilder'
        Rust : $'reqwest_wasm::RequestBuilder'
    }
:! request_builder =
    `(
        rust::fsharp_global "type reqwest_RequestBuilder = class end"
        $'' : $'backend_switch `(request_builder_switch)'
    )

:| request_type =
    | Get
    | Post

:== request =
    {
        url : string
        request_type : request_type
        body : string
        json : bool
        auto_refresh : bool
    }

:= new_request_get (url : string) : request_builder =
    := url = ~~ url
    := url = url |> sm'::to_std_string
    := url = ~~ url
    !\($'"reqwest_wasm::Client::builder().build().map_err(|err| err.to_string())?.get(!url)"')

:= new_request_post (url : string) : request_builder =
    := url = ~~ url
    := url = url |> sm'::to_std_string
    := url = ~~ url
    !\($'"reqwest_wasm::Client::builder().build().map_err(|err| err.to_string())?.post(!url)"')

:= request_send (request : request_builder) : async::future_pin (resultm::result' reqwest_response reqwest_error) =
    := request = ~~ request
    !\($'"Box::pin(reqwest_wasm::RequestBuilder::send(!request))"')

:= request_body (body : string) (request : request_builder) : request_builder =
    := body = body |> sm'::to_std_string
    !\\(body, $'"reqwest_wasm::RequestBuilder::body(!request, $0)"')

:= request_header (key : string) (value : string) (request : request_builder) : request_builder =
    := request = ~~ request
    := key = key |> sm'::to_std_string
    := value = value |> sm'::to_std_string
    !\\((key, value), $'"reqwest_wasm::RequestBuilder::header(!request, $0, $1)"')

:= request_json \* t. (obj : t) (request : request_builder) : request_builder =
    !\($'"reqwest_wasm::RequestBuilder::json(!request, &!obj)"')

:= response_text (response : reqwest_response) : async::future_pin (resultm::result' sm'::std_string reqwest_error) =
    !\($'"Box::pin(reqwest_wasm::Response::text(!response))"')

:! tcp_client =
    `(
        backend_switch `(()) `({}) {
            Fsharp = (\ () =>
                global """
                    type System_Net_Sockets_TcpClient = System.Net.Sockets.TcpClient

                    """
                        ) : () -> ()
        }
        $'' : $'System_Net_Sockets_TcpClient'
    )

:= new_tcp_client () : tcp_client =
    run_target \|
        | Fsharp => \ () => $'new `tcp_client ()'
        | _ => \ () => null ()

:! ip_address = $'System.Net.IPAddress'

:= ip_address_parse (s : string) : ip_address =
    s |> $'`ip_address.Parse'

:! tcp_listener = $'System.Net.Sockets.TcpListener'

:= new_tcp_listener (ip_address : ip_address) (port : i32) : tcp_listener =
    $'new `tcp_listener (!ip_address, !port)'

:= listener_start (listener : tcp_listener) : () =
    listener |> $'_.Start()'

:= listener_stop (listener : tcp_listener) : () =
    listener |> $'_.Stop()'

:= client_connect_async
    (host : string)
    (port : i32)
    (ct : threading::cancellation_token)
    (client : tcp_client)
    : async::value_task
    =
    run_target \|
        | Fsharp => \ () => $'!client.ConnectAsync (!host, !port, !ct)'
        | _ => \ () => null ()

:= main () =
    init_trace_state None